1. load the london full attractions
2. plot them on a map of london
3. try different clustering methods and parameters
4. visualise those clusters and pick 
(maybe NLP on the summaries and themes to help create useful labels for clusters)

In [2]:
import json
from pathlib import Path

import pandas as pd

DATA_FILE = Path("../data/london_full_attractions.json")

with open(DATA_FILE, "r", encoding="utf-8") as file:
    attractions = json.load(file)

df = pd.DataFrame(attractions)

print(f"Number of attractions: {len(df)}")
print(df.columns.tolist())

df.head()

Number of attractions: 404
['wikidata_id', 'name', 'category', 'description', 'latitude', 'longitude', 'image_url', 'sitelinks', 'summary', 'themes', 'interest_scores', 'recommended_visit_time', 'estimated_visit_mins', 'indoor', 'family_friendly', 'price_level']


,wikidata_id,name,category,description,latitude,longitude,image_url,sitelinks,summary,themes,interest_scores,recommended_visit_time,estimated_visit_mins,indoor,family_friendly,price_level
0,Q6373,British Museum,museum,"national museum in London, United Kingdom",51.519444,-0.126944,http://commons.wikimedia.org/wiki/Special:File...,108,The British Museum is a world-renowned museum ...,"[history, art, culture, ancient history]","{'history': 5, 'art': 5, 'architecture': 4, 'n...",morning,120,True,True,free
1,Q62378,Tower of London,museum,"castle in central London, United Kingdom",51.508200,-0.076198,http://commons.wikimedia.org/wiki/Special:File...,88,The Tower of London is a historic castle and m...,"[history, architecture, royalty, military hist...","{'history': 5, 'art': 3, 'architecture': 5, 'n...",morning,120,True,True,££
2,Q192988,Royal Observatory,museum,"observatory in Greenwich, London, UK",51.477833,-0.001389,http://commons.wikimedia.org/wiki/Special:File...,61,"The Royal Observatory in Greenwich, London, is...","[science, history, technology]","{'history': 4, 'art': 2, 'architecture': 3, 'n...",morning,60,True,True,£
3,Q674773,Science Museum,museum,"science museum in London, United Kingdom",51.497500,-0.174722,http://commons.wikimedia.org/wiki/Special:File...,47,The Science Museum in London is a world-renown...,"[science, technology, history, education]","{'history': 4, 'art': 2, 'architecture': 2, 'n...",morning,120,True,True,£
4,Q1990172,Sherlock Holmes Museum,museum,"museum in London, England",51.523694,-0.161000,http://commons.wikimedia.org/wiki/Special:File...,28,"The Sherlock Holmes Museum, located at 221B Ba...","[literature, history, entertainment]","{'history': 3, 'art': 2, 'architecture': 1, 'n...",morning,45,True,True,£


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   wikidata_id             404 non-null    object 
 1   name                    404 non-null    object 
 2   category                404 non-null    object 
 3   description             404 non-null    object 
 4   latitude                404 non-null    float64
 5   longitude               404 non-null    float64
 6   image_url               336 non-null    object 
 7   sitelinks               404 non-null    int64  
 8   summary                 404 non-null    object 
 9   themes                  404 non-null    object 
 10  interest_scores         404 non-null    object 
 11  recommended_visit_time  404 non-null    object 
 12  estimated_visit_mins    404 non-null    int64  
 13  indoor                  404 non-null    bool   
 14  family_friendly         404 non-null    bo

In [4]:
df[["latitude", "longitude"]].describe()

,latitude,longitude
count,404.000000,404.000000
mean,51.497008,-0.121095
std,0.076001,0.148856
min,51.281940,-0.541403
25%,51.455493,-0.187708
50%,51.505819,-0.121676
75%,51.534016,-0.066711
max,51.696240,0.286358


In [5]:
duplicate_coordinates = df.duplicated(
    subset=["latitude", "longitude"],
    keep=False,
)

print(
    f"Attractions sharing coordinates: "
    f"{duplicate_coordinates.sum()}"
)

df.loc[
    duplicate_coordinates,
    ["name", "latitude", "longitude"],
].sort_values(["latitude", "longitude"])

Attractions sharing coordinates: 0


,name,latitude,longitude


no attractions have exactly the same long and lat which is good

In [7]:
import folium

london_map = folium.Map(
    location=[51.5074, -0.1278],
    zoom_start=10,
    tiles="CartoDB positron",
)

for _, attraction in df.iterrows():
    folium.CircleMarker(
        location=[
            attraction["latitude"],
            attraction["longitude"],
        ],
        radius=4,
        popup=folium.Popup(
            attraction["name"],
            max_width=250,
        ),
        tooltip=attraction["name"],
        color="#2563eb",
        fill=True,
        fill_color="#2563eb",
        fill_opacity=0.7,
    ).add_to(london_map)

london_map
london_map.save("london_attractions_map.html")

go from long and lat to metres so clustering makes more sense

In [ ]:
import geopandas as gpd

attractions_gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(
        df["longitude"],
        df["latitude"],
    ),
    crs="EPSG:4326",
)

# Convert latitude/longitude into British National Grid coordinates
attractions_gdf = attractions_gdf.to_crs("EPSG:27700")

attractions_gdf["x"] = attractions_gdf.geometry.x
attractions_gdf["y"] = attractions_gdf.geometry.y

attractions_gdf[
    ["name", "latitude", "longitude", "x", "y"]
].head()